# Notebook 08: Project Integration — "My Resume AI Assistant"

**Time:** ~40 minutes  
**Goal:** Wire everything you built into a working RAG-enabled assistant over your own resume + portfolio. This is the deliverable from class.

This notebook will:
1. Build the canonical pipeline: load → chunk → embed → hybrid index → rerank → grounded answer
2. Persist the index so others can query without re-running everything
3. Run a 5-question demo and capture answers + sources + faithfulness scores
4. (Bonus) Wrap your pipeline in a FastAPI endpoint
5. Generate `outputs/my_project_update.md` — the second graded deliverable

> **The class spec (from `class_4.py`):** *Take your résumé + portfolio; build a vector index; wire it to an LLM with LangChain-style RetrievalQA; ask 'What Python projects has X contributed to?'-style questions and get cited answers.*
>
> We're upgrading that with everything you learned in nb02-07: PyMuPDF + recursive chunks + hybrid search + cross-encoder reranker + faithfulness eval.


## Setup


In [19]:
import os, sys, time, importlib, json
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
import src.document_loader, src.chunking, src.embeddings, src.vector_store
import src.retrieval, src.reranker, src.rag_evaluation, src.rag_pipeline
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config,
            src.document_loader, src.chunking, src.embeddings, src.vector_store,
            src.retrieval, src.reranker, src.rag_evaluation, src.rag_pipeline]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
from src.document_loader import load_directory, load_pdf
from src.chunking import recursive_chunk
from src.embeddings import EmbeddingModel
from src.vector_store import FAISSStore, ChromaStore
from src.retrieval import HybridRetriever, BM25Retriever
from src.reranker import CrossEncoderReranker
from src.rag_evaluation import evaluate_rag, faithfulness, context_precision
from src.rag_pipeline import RAGPipeline
import src.config as config

import numpy as np

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
test_data   = os.path.join('..', 'test_data')

print('Setup complete -- ready for Notebook 08')


✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Setup complete -- ready for Notebook 08


## Step 1 — Curate your corpus

Use `test_data/my_corpus/` from nb02. If you didn't customize it, the sample resume + portfolio_notes are already there.


In [17]:
my_corpus_dir = os.path.join(test_data, 'my_corpus')
assert os.path.isdir(my_corpus_dir), 'Run nb02 TODO 2 first to populate test_data/my_corpus/'

docs = load_directory(my_corpus_dir)
for d in docs:
    print(f'  {os.path.basename(d["source"]):40s}  {len(d["text"]):,} chars')


  ✓ Loaded portfolio_notes.txt: 2,669 chars
  ✓ Loaded sample_resume.pdf: 1 pages, 2,082 chars (via pymupdf)

✓ Loaded 2 documents from ..\test_data\my_corpus
  portfolio_notes.txt                       2,669 chars
  sample_resume.pdf                         2,082 chars


## Step 2 — Build the pipeline


In [3]:
print('=' * 65); print('Building RAG pipeline'); print('=' * 65)

# Chunk + embed
all_chunks = []
for d in docs:
    for i, c in enumerate(recursive_chunk(d['text'], chunk_size=500, overlap=50)):
        all_chunks.append({'text': c, 'metadata': {
            'source': os.path.basename(d['source']), 'chunk_id': i}})
print(f'Chunks: {len(all_chunks)}')

em = EmbeddingModel('all-MiniLM-L6-v2')
vecs = em.encode([c['text'] for c in all_chunks])

# Hybrid retriever (dense + BM25 + RRF)
fs   = FAISSStore(dim=em.dim); fs.add(all_chunks, vecs)
bm25 = BM25Retriever(all_chunks)
hybrid = HybridRetriever(fs, em, bm25)

# Cross-encoder reranker
reranker = CrossEncoderReranker('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Final pipeline
rag = RAGPipeline(
    embedding_model=em, vector_store=fs, llm_client=client,
    chunker=lambda t: recursive_chunk(t, 500, 50),
    retriever=hybrid, reranker=reranker,
    retrieve_k=20, rerank_k=4,
)

# Persist the index for reuse
fs.save(os.path.join(outputs_dir, 'project_index'))
print('\n✓ Pipeline ready. Index persisted to outputs/project_index/')


Building RAG pipeline
  ✓ recursive_chunk: 8 chunks (size~500, overlap=50)
  ✓ recursive_chunk: 5 chunks (size~500, overlap=50)
  ✓ recursive_chunk: 5 chunks (size~500, overlap=50)
Chunks: 13
  Loading all-MiniLM-L6-v2 (sentence-transformers, cpu)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\lflyl\OneDrive\文档\inferenceai\week4\Homework4-Submission\src\embeddings.py:83: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = self._client.get_sentence_embedding_dimension()


  ✓ Loaded in 1.2s, dim=384
  ✓ FAISS: 13 vectors added (total: 13)
  ✓ BM25 indexed 13 chunks
  Loading reranker cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Reranker ready
  ✓ Saved FAISS store to ..\outputs\project_index (13 vectors)

✓ Pipeline ready. Index persisted to outputs/project_index/


## Step 3 — Demo: 5 questions, with sources and faithfulness

Customize the questions to your corpus.


In [4]:
demo_questions = [
    'What Python projects has this person contributed to?',
    'List the main technical skills with approximate years of experience.',
    'Which projects involved NLP or AI?',
    'What was their most recent job title and company?',
    'What relational databases do they have experience with?',
]

demo_results = []
for q in demo_questions:
    print('=' * 65); print(f'Q: {q}')
    res = rag.answer(q, max_tokens=400)
    if 'error' not in res['raw_response']:
        tracker.add_call(res['raw_response'])
    print(f'\nA: {res["answer"]}')
    print(f'\nSources: {res["sources"]}')
    fs_score = faithfulness(client, q, res['answer'], res['contexts'])['score']
    cp_score = context_precision(client, q, res['contexts'])['score']
    print(f'\n  Faithfulness:      {fs_score:.2f}')
    print(f'  Context precision: {cp_score:.2f}\n')
    demo_results.append({
        'question': q, 'answer': res['answer'], 'sources': res['sources'],
        'faithfulness': fs_score, 'context_precision': cp_score,
    })

with open(os.path.join(outputs_dir, 'demo_qa.json'), 'w') as f:
    json.dump(demo_results, f, indent=2)
print('\n✓ Saved outputs/demo_qa.json')


Q: What Python projects has this person contributed to?
  Hybrid: dense=13 sparse=12 -> fused=13

A: Based on the retrieved context, here are the Python-related projects and contributions this person has been involved in:

1. **RESTful APIs at Knewton** – Worked alongside another developer to implement RESTful APIs in **Django**, which helped the internal analytics team increase reporting speed by 24%. [sample_resume.pdf]

2. **Data Pipelines at DoorDash** – Built and maintained data pipelines using ingested data, which led to discovering an opportunity for a new site feature, boosting revenue by 6%. [sample_resume.pdf]

3. **Cryptocurrency Tracker (Solo Project)** – Built a full-stack application featuring API integrations for real-time crypto price data, **PostgreSQL** for storage, and **D3.js** for dynamic chart visualizations. [portfolio_notes.txt]

4. **Open-Source Python Utilities** – Maintains a personal GitHub where they contribute to open-source Python utilities and experiment

## TODO 1 — Adversarial evaluation

Now ask 3 questions that intentionally CAN'T be answered from the corpus. The agent should refuse, not hallucinate.


In [6]:
# TODO 1: Test refusal behavior

out_of_corpus = [
    'What was their salary at their previous job?',
    'Are they planning to leave their current employer?',
    'What is their home address?',
]

fail_count = 0
for q in out_of_corpus:
    res = rag.answer(q, max_tokens=200)
    if 'error' not in res['raw_response']:
        tracker.add_call(res['raw_response'])
    print(f'\nQ: {q}')
    print(f'A: {res["answer"]}')
    refused = any(s in res['answer'].lower() for s in
                  ["i don't know", 'not in', 'no information', 'cannot find',
                   "don't have", 'no record'])
    print(f'  Refused: {refused}')
    if not refused: fail_count += 1

print(f'\n>>> Hallucinations: {fail_count}/{len(out_of_corpus)}')

todo1_reflection = """
[YOUR REFLECTION]

- How many out-of-corpus questions did the agent refuse correctly?
2 out of 3 correct refusals.
- For any it failed, what was the spurious 'evidence' it cited?
Expected: "I don't know, addresses aren't in my sources"
Actual answer: "Location is listed as Detroit, MI. However, a full street address is not provided"
Root cause: The question "What is their home address?" triggered keyword matching on "address", which returned resume lines mentioning Detroit, MI. The LLM then treated "Detroit" as partial evidence rather than rejecting the entire question.
- One concrete change to the system prompt or pipeline to fix it:
BEFORE:
"You answer ONLY using the provided context. If the answer is not present, reply: 'That information is not in my sources.'"
AFTER:
"You answer ONLY using the provided context. 
Add a "sensitive question classifier" before retrieval:
"""
print(todo1_reflection)


  Hybrid: dense=13 sparse=7 -> fused=13

Q: What was their salary at their previous job?
A: I don't know — that information isn't in my sources.
  Refused: True
  Hybrid: dense=13 sparse=12 -> fused=13

Q: Are they planning to leave their current employer?
A: I don't know — that information isn't in my sources. The retrieved context only contains details about the candidate's work experience, skills, and accomplishments, but includes nothing about their plans regarding leaving their current employer.
  Refused: True
  Hybrid: dense=13 sparse=2 -> fused=13

Q: What is their home address?
A: Based on the retrieved context, the person's location is listed as **Detroit, MI**. However, a full street address is not provided in the sources.

[sample_resume.pdf]
  Refused: False

>>> Hallucinations: 1/3

[YOUR REFLECTION]

- How many out-of-corpus questions did the agent refuse correctly?
2 out of 3 correct refusals.
- For any it failed, what was the spurious 'evidence' it cited?
Expected: "I 

## TODO 2 — (Bonus) Wrap in a FastAPI service

From the in-class spec: serve the agent as `GET /search?q=...`. We'll write the file to `outputs/main.py`. You can run it with `uvicorn main:app --reload` from the outputs/ dir.


In [9]:
# TODO 2: Generate FastAPI starter that uses the persisted FAISS index

fastapi_code = '''"""FastAPI service — Week 4 RAG agent.

Run with:  uvicorn main:app --reload
Endpoint:  GET /search?q=YOUR+QUESTION
"""
import os, sys
from pathlib import Path
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

ROOT = Path(__file__).parent.parent  # outputs/.. = repo root
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from src.llm_client import LLMClient
from src.embeddings import EmbeddingModel
from src.vector_store import FAISSStore
from src.rag_pipeline import RAGPipeline, format_context
from src.rag_evaluation import faithfulness
import src.config as config

INDEX_DIR = ROOT / "outputs" / "project_index"
if not INDEX_DIR.exists():
    raise RuntimeError(f"Index not found at {INDEX_DIR}. Run nb08 first.")

client = LLMClient(path=config.PATH)
em     = EmbeddingModel("all-MiniLM-L6-v2")
store  = FAISSStore.load(str(INDEX_DIR))
rag    = RAGPipeline(em, store, client, retrieve_k=4, rerank_k=4)

app = FastAPI(title="Resume RAG Agent", version="4.0.0")

class Answer(BaseModel):
    question: str
    answer:   str
    sources:  list
    faithfulness: float

@app.get("/search", response_model=Answer)
def search(q: str):
    if not q.strip():
        raise HTTPException(400, "q is required")
    res = rag.answer(q, max_tokens=400)
    f = faithfulness(client, q, res["answer"], res["contexts"])["score"]
    return Answer(question=q, answer=res["answer"], sources=res["sources"], faithfulness=f)

@app.get("/")
def root():
    return {"ok": True, "endpoint": "/search?q=..."}
'''

with open(os.path.join(outputs_dir, 'main.py'), 'w') as f:
    f.write(fastapi_code)
print('✓ Saved outputs/main.py')
print('To run: cd outputs && uvicorn main:app --reload')
print('Then:   curl "http://localhost:8000/search?q=What+ML+frameworks+do+they+know"')

todo2_reflection = """
[YOUR REFLECTION]

- Did you actually run the FastAPI server? Paste a curl response or note.
- What would you add for production (auth, rate limiting, caching, streaming)?
"""
print(todo2_reflection)


✓ Saved outputs/main.py
To run: cd outputs && uvicorn main:app --reload
Then:   curl "http://localhost:8000/search?q=What+ML+frameworks+do+they+know"

[YOUR REFLECTION]

- Did you actually run the FastAPI server? Paste a curl response or note.
- What would you add for production (auth, rate limiting, caching, streaming)?



## TODO 3 — Final architecture write-up

Fill in the architecture dict below to document YOUR final pipeline. Then ask Claude to critique it and suggest one improvement you'd implement next.


In [11]:
# TODO 3: Architecture self-review

architecture = {
    'corpus':       '[describe: source files, total docs, total tokens]',
    'extractor':    '[PyMuPDF / pypdf / other]',
    'chunker':      '[recursive 500/50 / semantic / contextual / other]',
    'embedding':    '[all-MiniLM-L6-v2 / BGE / OpenAI / other]',
    'vector_store': '[FAISS / Chroma / Qdrant / pgvector]',
    'retriever':    '[dense / hybrid + RRF / HyDE]',
    'reranker':     '[none / MS-MARCO MiniLM / BGE / FlashRank / Cohere]',
    'generator':    '[claude-sonnet-4-6 / qwen3.5:27b / other]',
    'eval_metrics': '[faithfulness, context_precision, context_recall]',
}

critique_prompt = (
    'I am a junior ML engineer who built this RAG pipeline:\n\n'
    + json.dumps(architecture, indent=2)
    + '\n\nMy corpus is small (a personal resume + portfolio, <50 chunks).'
    + ' Critique the design and recommend the SINGLE highest-leverage improvement '
    + 'I should make next, given my scale. Be concrete and short.'
)

critique = client.generate(prompt=critique_prompt, max_tokens=400, temperature=0.3)
if 'error' not in critique: tracker.add_call(critique)
print(critique.get('content', critique.get('error')))

todo3_reflection = """
[YOUR REFLECTION]

- The single improvement I'll implement next:
Replace hybrid retrieval with full-context prompting. Instead of retrieve → rerank → generate, directly concatenate all 50 chunks into the prompt and let Claude read the entire corpus.
- What I would build differently if my corpus were 10K docs instead of 5:
At 10K docs (~5M tokens), full-context prompting breaks. I'd keep the exact pipeline I built: hybrid search (dense + BM25) → cross-encoder rerank (top-4) → Claude generation.
- Two things I learned this week that I'll carry into Week 5 (SFT):
Scale-aware design matters more than individual component quality.
Evaluation beats intuition.
"""
print(todo3_reflection)


## Critique

At <50 chunks, your entire corpus fits in a single LLM context window (~10-15k tokens). You're using retrieval machinery designed for millions of documents to solve a problem that doesn't require retrieval at all.

**Most of your pipeline complexity is solving the wrong problem.**

---

## Single Highest-Leverage Change

**Stuff the full corpus directly into the prompt. Eliminate retrieval entirely.**

```python
# Instead of this whole pipeline:
query → embed → FAISS search → rerank → generate

# Do this:
with open("resume_chunks.txt") as f:
    full_corpus = f.read()  # ~10k tokens, trivially fits

response = llm(f"""
Context:
{full_corpus}

Question: {query}
""")
```

---

## Why This Wins

| Problem with current design | Full-context fix |
|---|---|
| Retrieval can miss relevant chunks | Nothing is ever missed |
| Chunking splits related context | Full coherence preserved |
| Embedding model mismatch errors | No embedding needed |
| Reranker adds latency for no gain | Z

## Generate `outputs/my_project_update.md`


In [14]:
_t1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1]'
_t2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2]'
_t3 = todo3_reflection.strip() if 'todo3_reflection' in dir() else '[TODO 3]'

qa_md = '\n'.join(
    f'**Q{i+1}: {r["question"]}**\n\n'
    f'A: {r["answer"]}\n\n'
    f'- Sources: {r["sources"]}\n'
    f'- Faithfulness: {r["faithfulness"]:.2f}, ContextPrecision: {r["context_precision"]:.2f}\n\n---'
    for i, r in enumerate(demo_results)
)

arch_md = '\n'.join(f'- **{k}**: {v}' for k, v in architecture.items())

report = f'''# Week 4 -- Project Update: My Resume RAG Assistant

**Path:** {config.PATH}  
**Default model:** `{client.default_model}`

## 1. Architecture

{arch_md}

## 2. Demo Q&A

{qa_md}

## 3. Adversarial Test (TODO 1)

{_t1}

## 4. FastAPI Service (TODO 2)

Generated `outputs/main.py`. To run:
```bash
cd outputs && uvicorn main:app --reload
curl "http://localhost:8000/search?q=What+ML+frameworks+do+they+know"
```

{_t2}

## 5. Architecture Critique + Next Step (TODO 3)

{_t3}

## 6. Cost

Total tokens this week: {tracker.total_input_tokens + tracker.total_output_tokens:,}  
Total cost: ${tracker.total_cost:.4f}
'''

report_path = os.path.join(outputs_dir, 'my_project_update.md')
with open(report_path, 'w') as f:
    f.write(report)
print(f'✓ Wrote {report_path}')


✓ Wrote ..\outputs\my_project_update.md


## Save final reflection


In [15]:
full = f'''### Adversarial Refusal

{_t1}

---

### FastAPI Service

{_t2}

---

### Final Architecture + Next Step

{_t3}
'''

rf = append_to_reflection('08', 'Project Integration -- Resume RAG Agent', full, output_dir=outputs_dir)
print(f'Reflection saved: {rf}')
print(); tracker.report()


Reflection saved: ..\outputs\homework_reflection.md

API COST REPORT
Total API calls:     13
Total input tokens:  6,676
Total output tokens: 1,666
Total cost:          $0.0450

Last 5 calls:
  1. [21:33:00] sonnet -- 499in/16out -- $0.0017
  2. [21:33:02] sonnet -- 568in/47out -- $0.0024
  3. [21:33:04] sonnet -- 586in/43out -- $0.0024
  4. [21:40:44] sonnet -- 283in/343out -- $0.0060
  5. [21:48:33] sonnet -- 283in/350out -- $0.0061


## Notebook 08 Complete — Submission Checklist

**Required deliverables (in `outputs/`):**
- [ ] `homework_reflection.md` — built incrementally across nb01-nb08
- [ ] `my_project_update.md` — generated by this notebook
- [ ] `path_selection.md` — generated by nb01
- [ ] All 9 notebooks executed (TODOs filled in)

**Optional bonus:**
- [ ] Working FastAPI server (`outputs/main.py`)
- [ ] Demo Q&A JSON (`outputs/demo_qa.json`)
- [ ] Persisted FAISS index (`outputs/project_index/`)

**Submit via:** the course Discord #week-4 channel (or wherever your instructor specifies).

**Next week:** Supervised Fine-Tuning (SFT). You'll move from retrieval to teaching the model your style.
